In [ ]:
from robotdesigner.mujoco.composer import build_arm
import gymnasium as gym

import mujoco as mj
import mjinx


In [ ]:
gym.register(
        id="AbstractArm-v0",
        entry_point="robotdesigner.envs.abstract_mujoco_env:make_env",
    )

In [ ]:
spec = build_arm(ndof=3, control="pos")

In [ ]:
env = gym.make(
    "AbstractArm-v0",
    spec=spec,
    render_mode="rgb_array",
)

In [ ]:
import jax
import numpy as np
from mujoco import mjx
from mjinx.problem import Problem
from mjinx.components.tasks import FrameTask
from mjinx.components.barriers import JointBarrier
from mjinx.solvers import LocalIKSolver
from mjinx.configuration import integrate

In [ ]:
import mujoco_warp as mjw

In [ ]:
import mujoco_playground

In [ ]:
# jax kernel choose between jax and warp

In [ ]:
mj_model = spec.compile()
# mjx_model = mjx.put_model(mj_model)

In [ ]:
mjx_model = mjx.put_model(mj_model)

In [ ]:
problem = Problem(mjx_model)

In [ ]:
frame_task = FrameTask("ee_task", cost=1, gain=20, obj_name="link3_link")
problem.add_component(frame_task)

In [ ]:
joints_barrier = JointBarrier("jnt_range", gain=10)
problem.add_component(joints_barrier)

In [ ]:
solver = LocalIKSolver(mjx_model)

# Initial configuration
dt = 1e-2

In [ ]:
solve_jit = jax.jit(solver.solve)
integrate_jit = jax.jit(integrate, static_argnames=["dt"])

In [ ]:
q = np.zeros(mjx_model.nq)
solver_data = solver.init(q)
for t in np.arange(0, 5, dt):
    # Update target and compile problem
    frame_task.target_frame = np.array([0.1 * np.sin(t), 0.1 * np.cos(t), 0.1, 1, 0, 0, 0])
    problem_data = problem.compile()

    # Solve the IK problem
    opt_solution, solver_data = solve_jit(q, solver_data, problem_data)

    # Integrate to get new configuration
    q = integrate_jit(
        mjx_model,
        q,
        opt_solution.v_opt,
        dt,
    )
    print(f"Time: {t:.2f}, q: {q}")

In [ ]:
from types import SimpleNamespace
class Experiment(SimpleNamespace):
    def __init__(self):
        super().__init__()
        q: int = 0

e = Experiment()

In [ ]:
e.q = 4

In [ ]:
e.q = 2